In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool as langchain_tool
from langchain.agents import create_agent

# DeepSeek exposes an OpenAI-compatible API.
llm = ChatOpenAI(
   model=os.environ["DEEPSEEK_MODEL_ID"],
   api_key=os.environ["DEEPSEEK_API_KEY"],
   base_url=os.environ["DEEPSEEK_BASE_URL"],
   temperature=0,
)
print(f"✅ Language model initialized: {llm.model_name}")

# --- Define a Tool ---
@langchain_tool
def search_information(query: str) -> str:
   """
   Provides factual information on a given topic. Use this tool to find answers to phrases
   like 'capital of France' or 'weather in London?'.
   """
   print(f"\n--- 🛠️ Tool Called: search_information with query: '{query}' ---")
   # Simulate a search tool with a dictionary of predefined results.
   simulated_results = {
       "weather in london": "The weather in London is currently cloudy with a temperature of 15°C.",
       "capital of france": "The capital of France is Paris.",
       "population of earth": "The estimated population of Earth is around 8 billion people.",
       "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
       "default": f"Simulated search result for '{query}': No specific information found, but the topic seems interesting."
   }
   result = simulated_results.get(query.lower(), simulated_results["default"])
   print(f"--- TOOL RESULT: {result} ---")
   return result

tools = [search_information]

# --- Create a LangChain 1.x Tool-Calling Agent ---
agent = create_agent(
   model=llm,
   tools=tools,
   system_prompt="You are a helpful assistant.",
)

def run_agent_with_tool(query: str):
   """Invokes the agent with a query and prints the final response."""
   print(f"\n--- 🏃 Running Agent with Query: '{query}' ---")
   response = agent.invoke({
       "messages": [{"role": "user", "content": query}]
   })
   print("\n--- ✅ Final Agent Response ---")
   print(response["messages"][-1].content)

for query in (
   "What is the capital of France?",
   "What's the weather like in London?",
   "Tell me something about dogs.",
):
   run_agent_with_tool(query)

✅ Language model initialized: deepseek-v4-flash

--- 🏃 Running Agent with Query: 'What is the capital of France?' ---

--- 🛠️ Tool Called: search_information with query: 'capital of France' ---
--- TOOL RESULT: The capital of France is Paris. ---

--- ✅ Final Agent Response ---
The capital of France is **Paris**.

--- 🏃 Running Agent with Query: 'What's the weather like in London?' ---

--- 🛠️ Tool Called: search_information with query: 'weather in London' ---
--- TOOL RESULT: The weather in London is currently cloudy with a temperature of 15°C. ---

--- ✅ Final Agent Response ---
The weather in London is currently **cloudy** with a temperature of **15°C**.

--- 🏃 Running Agent with Query: 'Tell me something about dogs.' ---

--- 🛠️ Tool Called: search_information with query: 'facts about dogs' ---
--- TOOL RESULT: Simulated search result for 'facts about dogs': No specific information found, but the topic seems interesting. ---

--- ✅ Final Agent Response ---
The search didn't turn up